# 05 · Shots
Every shot's tracking frame, oriented so the shooting team attacks toward +x. A first look
at where defenders and keepers are when shots are taken.

In [ ]:
import sys
from pathlib import Path

# Make the repo's src/ importable no matter where the notebook is run from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
from src.config import PROCESSED_DIR
from src.loading import build_shot_frames, check_goals
from src.pitch import draw_pitch, plot_frame

if (PROCESSED_DIR / "shots.parquet").exists():
    shots = pd.read_parquet(PROCESSED_DIR / "shots.parquet")
    frames = pd.read_parquet(PROCESSED_DIR / "shot_frames.parquet")
else:
    shots, frames = build_shot_frames(verbose=False)
t = shots[shots.has_tracking]
print(f"{len(shots)} shots, {int(shots.is_goal.sum())} goals, {len(t)} with tracking")

## Data checks

In [ ]:
print(f"orientation check passed: {t.orientation_ok.mean():.1%}")
print(f"keeper inferred:          {int(t.gk_inferred.sum())}")
print(f"ball missing:             {t.ball_x.isna().mean():.1%}")
print(f"frame gap used (not 0):   {(t.frame_gap != 0).mean():.1%}")
goals = check_goals()
print(f"matches reconciling with the score: {goals.matches_score.sum()} / {len(goals)}")
goals[~goals.matches_score]

## Where shots come from

In [ ]:
ax = draw_pitch(half=True)
ax.scatter(t.shooter_x, t.shooter_y, s=12, alpha=0.5, label="shot")
g = t[t.is_goal]
ax.scatter(g.shooter_x, g.shooter_y, s=40, marker="*", c="#00a82f", label="goal")
ax.legend(loc="lower left", fontsize=8)
plt.show()

t["distance"] = np.hypot(t.pitch_length / 2 - t.shooter_x, t.shooter_y)
t.groupby(pd.cut(t.distance, [0, 6, 11, 16.5, 25, 60]), observed=True).agg(
    shots=("shot_id", "size"), goals=("is_goal", "sum"), rate=("is_goal", "mean")).round(3)

## Defenders at the moment of the shot

In [ ]:
def in_triangle(px, py, ax_, ay, bx, by, cx, cy):
    d1 = (px - bx) * (ay - by) - (ax_ - bx) * (py - by)
    d2 = (px - cx) * (by - cy) - (bx - cx) * (py - cy)
    d3 = (px - ax_) * (cy - ay) - (cx - ax_) * (py - ay)
    neg = (d1 < 0) | (d2 < 0) | (d3 < 0)
    pos = (d1 > 0) | (d2 > 0) | (d3 > 0)
    return ~(neg & pos)

f = frames.merge(t[["shot_id", "shooter_x", "shooter_y", "pitch_length", "is_goal"]], on="shot_id")
L = f.pitch_length / 2
f["in_cone"] = in_triangle(f.x, f.y, f.shooter_x, f.shooter_y, L, -3.66, L, 3.66)
f["dist_to_shooter"] = np.hypot(f.x - f.shooter_x, f.y - f.shooter_y)
d = f[f.role == "defender"]
per_shot = d.groupby("shot_id").agg(def_in_cone=("in_cone", "sum"),
                                    nearest_def=("dist_to_shooter", "min"),
                                    share_extrapolated=("is_detected", lambda s: 1 - s.fillna(False).astype(bool).mean()))
per_shot = per_shot.join(t.set_index("shot_id")[["is_goal"]])
per_shot.groupby(per_shot.def_in_cone.clip(upper=4)).agg(shots=("is_goal", "size"), goal_rate=("is_goal", "mean")).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(per_shot.nearest_def.clip(upper=10), bins=30); axes[0].set_title("Nearest defender to shooter (m)")
axes[1].hist(per_shot.share_extrapolated, bins=11); axes[1].set_title("Share of defenders extrapolated, per shot")
plt.tight_layout()

## Shot frames
Hollow markers are extrapolated players.

In [ ]:
sample = pd.concat([t[t.is_goal].head(3), t[~t.is_goal].head(3)])
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, s in zip(axes.flat, sample.itertuples()):
    plot_frame(frames[frames.shot_id == s.shot_id], ax=ax, length=s.pitch_length, width=s.pitch_width,
               half=True, color_by="role",
               title=f"{s.team_shortname} {s.minute_start}' {'GOAL' if s.is_goal else ''}")
plt.tight_layout()